# 🏨 Hotel Cancellation Classification
**เป้าหมาย:** สร้างโมเดล Classification เพื่อทำนาย `cancelled` (0 = ไม่ยกเลิก, 1 = ยกเลิก)

Workflow: **Data Understanding → Cleaning → EDA → Preprocessing → Train/Test Split → Modeling → Evaluation → Select Best Model → Save Model → Deploy**

> จุดสำคัญ: Dataset มี Class Imbalance จึงไม่ควรตัดสินโมเดลจาก Accuracy เพียงอย่างเดียว โดยจะดู Precision, Recall, F1-score, ROC-AUC และ Confusion Matrix ร่วมกัน


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    roc_auc_score
)
import joblib

pd.set_option("display.max_columns", None)


## 1) Load & Understand Data
ตรวจขนาดข้อมูล ชนิดข้อมูล Missing Value, Duplicate, ค่าของ Category และสถิติเบื้องต้น


In [ ]:
FILE = "C09_hotel_cancellation_classification.xlsx"
df = pd.read_excel(FILE)

print("Shape:", df.shape)
display(df.head())
df.info()


In [ ]:
print("Missing values:")
display(df.isna().sum().to_frame("missing"))

print("Exact duplicate rows:", df.duplicated().sum())
print("Target distribution:")
display(df["cancelled"].value_counts())
display(df.describe(include="all"))


In [ ]:
for col in ["deposit_paid", "room_type"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


### สิ่งที่พบจากไฟล์จริง
- ข้อมูลเริ่มต้น 1,062 แถว 8 คอลัมน์
- มี Exact duplicate 12 แถว
- มี Missing Value หลายคอลัมน์
- `deposit_paid` มีรูปแบบไม่สม่ำเสมอ เช่น `Yes`, `YES`, `yes`, ช่องว่าง
- `room_type` มีตัวพิมพ์/ช่องว่างไม่สม่ำเสมอ
- ตัวแปรที่ควรเป็นจำนวนเต็มมีค่าทศนิยมผิดปกติบางแถว
- `lead_days` ปกติอยู่ช่วง 1–120 แต่พบค่าผิดปกติ 131.2 และ 182.4
- Target ไม่สมดุล: Class 0 มีมากกว่า Class 1


## 2) Data Cleaning


In [ ]:
data = df.drop_duplicates().copy()

# ทำ Category ให้เป็นมาตรฐาน
data["deposit_paid"] = data["deposit_paid"].apply(
    lambda x: x.strip().title() if isinstance(x, str) else np.nan
)
data["room_type"] = data["room_type"].apply(
    lambda x: x.strip().title() if isinstance(x, str) else np.nan
)

# คอลัมน์เหล่านี้ควรเป็นจำนวนเต็ม หากพบ decimal ให้ถือเป็น invalid -> NaN
for col in ["nights", "guests", "special_requests"]:
    mask = data[col].notna() & (data[col] % 1 != 0)
    data.loc[mask, col] = np.nan

# จากข้อมูลปกติ lead_days อยู่ 1-120; ค่ามากกว่า 120 ในไฟล์เป็นค่าผิดปกติ
data.loc[data["lead_days"] > 120, "lead_days"] = np.nan

print("After removing duplicates:", data.shape)
display(data.isna().sum().to_frame("missing_after_cleaning"))
print(data["deposit_paid"].value_counts(dropna=False))
print(data["room_type"].value_counts(dropna=False))


**เหตุผลที่ยังไม่ Fill Missing ตอนนี้:** เพื่อป้องกัน Data Leakage เราจะ Split Train/Test ก่อน แล้วให้ `SimpleImputer` เรียน Median/Mode จาก **Training set เท่านั้น** ภายใน Pipeline


## 3) EDA / Visualization


In [ ]:
target_counts = data["cancelled"].value_counts().sort_index()
ax = target_counts.plot(kind="bar", title="Cancellation Class Distribution")
ax.set_xlabel("cancelled (0=No, 1=Yes)")
ax.set_ylabel("Bookings")
plt.show()

print(data["cancelled"].value_counts(normalize=True).rename("proportion"))


In [ ]:
deposit_rate = pd.crosstab(data["deposit_paid"], data["cancelled"], normalize="index")
display(deposit_rate)

deposit_rate[1].plot(kind="bar", title="Cancellation Rate by Deposit")
plt.ylabel("Cancellation rate")
plt.show()


In [ ]:
room_rate = pd.crosstab(data["room_type"], data["cancelled"], normalize="index")
display(room_rate)

room_rate[1].plot(kind="bar", title="Cancellation Rate by Room Type")
plt.ylabel("Cancellation rate")
plt.show()


In [ ]:
display(data.groupby("cancelled")[["lead_days","nights","guests","special_requests"]].mean())

data.boxplot(column="lead_days", by="cancelled")
plt.suptitle("")
plt.title("Lead Days by Cancellation")
plt.ylabel("lead_days")
plt.show()


### EDA Interpretation
จากไฟล์นี้:
- กลุ่มยกเลิกมีสัดส่วนประมาณ 24% ทำให้เกิด Class Imbalance
- ลูกค้าที่ **ไม่จ่ายมัดจำ** มีอัตรายกเลิกสูงกว่ากลุ่มที่จ่ายมัดจำอย่างชัดเจน
- ค่าเฉลี่ย `lead_days` ของกลุ่มยกเลิกสูงกว่ากลุ่มไม่ยกเลิก
- `room_type` ทั้ง 3 กลุ่มมีอัตรายกเลิกใกล้เคียงกัน จึงอาจมีอิทธิพลน้อยกว่า Deposit/Lead days


## 4) Feature / Target + Train/Test Split


In [ ]:
X = data.drop(columns=["booking_id", "cancelled"])
y = data["cancelled"]

# booking_id เป็น Identifier จึงไม่ใช้เป็น Feature
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("\nTrain target proportion:")
print(y_train.value_counts(normalize=True))
print("\nTest target proportion:")
print(y_test.value_counts(normalize=True))


## 5) Preprocessing Pipeline
- Numeric: Median Imputation + StandardScaler
- Categorical: Most-frequent Imputation + One-Hot Encoding
- ใช้ Pipeline เพื่อให้ขั้นตอนเดียวกันถูกนำไปใช้กับ Test data และตอน Deploy


In [ ]:
numeric_features = ["lead_days", "nights", "guests", "special_requests"]
categorical_features = ["deposit_paid", "room_type"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])


## 6) Train 3 Models
ใช้ `class_weight="balanced"` เพื่อให้โมเดลให้ความสำคัญกับ Class 1 ที่มีจำนวนน้อยขึ้น


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=10,
        class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=7, min_samples_leaf=5,
        class_weight="balanced", random_state=42
    )
}

fitted_models = {}
results = []

for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    prob = pipe.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC_AUC": roc_auc_score(y_test, prob),
        "TN": tn, "FP": fp, "FN": fn, "TP": tp
    })
    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df.style.format({
    "Accuracy":"{:.3f}", "Precision":"{:.3f}", "Recall":"{:.3f}",
    "F1":"{:.3f}", "ROC_AUC":"{:.3f}"
}))


## 7) Evaluation
Accuracy อย่างเดียวอาจทำให้เข้าใจผิด เพราะถ้าทายทุกแถวเป็น Class 0 ก็ได้ Accuracy ราว 76% จากความไม่สมดุลของข้อมูล แต่ไม่สามารถจับลูกค้าที่จะยกเลิกได้เลย

ดังนั้นโปรเจกต์นี้ให้ความสำคัญกับ **Recall ของ Class 1** และ **F1-score** ร่วมกับ Confusion Matrix


In [ ]:
for name, pipe in fitted_models.items():
    pred = pipe.predict(X_test)
    print("="*70)
    print(name)
    print(classification_report(y_test, pred, digits=3))
    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(name)
    plt.show()


### ผลจากการรันด้วย `random_state=42`
ค่าประมาณที่ควรได้:
- Logistic Regression: Accuracy ≈ 0.657, Recall ≈ 0.580
- Decision Tree: Accuracy ≈ 0.643, Recall ≈ 0.620
- Random Forest: Accuracy ≈ 0.695, Recall ≈ 0.580

Decision Tree จับ Class 1 ได้มากที่สุดในรอบทดสอบนี้ ส่วน Random Forest ให้สมดุล Accuracy/Precision/F1 ที่ดีกว่าโดยรวม จึงเลือก **Random Forest** เป็นโมเดล Deploy ในเวอร์ชันนี้

> ไม่ควรอ้างว่าโมเดล “แม่นยำ 69.5%” อย่างเดียว เพราะ baseline ของ Class 0 สูงกว่า Accuracy ของโมเดล balanced ได้ สิ่งที่โมเดลทำเพิ่มคือพยายามตรวจจับกลุ่ม Cancelled ซึ่งเป็นเป้าหมายที่สำคัญทางธุรกิจ


## 8) Save Final Model


In [ ]:
best_model = fitted_models["Random Forest"]
joblib.dump(best_model, "hotel_model.pkl")
print("Saved: hotel_model.pkl")


## 9) Deployment
ไฟล์ `app.py` ที่แนบในโปรเจกต์จะโหลด `hotel_model.pkl` และรับค่า:
`lead_days`, `nights`, `guests`, `special_requests`, `deposit_paid`, `room_type`

รันด้วย:
```bash
pip install -r requirements.txt
streamlit run app.py
```

### สรุปสำหรับนำเสนอ
โปรเจกต์นี้ทำเพื่อทำนายลูกค้าที่มีแนวโน้มยกเลิกการจอง เพื่อช่วยโรงแรมวางแผนห้องพักและลดความเสี่ยงด้านรายได้ ข้อมูลถูกทำความสะอาดและป้องกัน Data Leakage ด้วย Pipeline จากนั้นเปรียบเทียบ 3 โมเดลด้วยหลาย Metrics และ Deploy โมเดลที่เลือกผ่าน Streamlit
